# Immune Health Atlas: multiresolution sample representations

**Development install:** from this repository’s root run `pip install -e .` so the notebook matches the `CellGroupComposition` API (including `apply_clr`).

Tools such as **scECODA** summarize each sample as compositional vectors and analyze them after a **centered log-ratio (CLR)** transform so that Euclidean distances respect relative abundances. **The same construction is available in patpy**: `CellGroupComposition` aggregates cell groups per sample, optional `apply_clr=True` applies CLR with a pseudocount, and pairwise sample distances are Euclidean in that space. We therefore **do not add scECODA as a separate method** in patpy—the benchmark below uses `CellGroupComposition(apply_clr=True)` as the CLR-compositional baseline. See the `CellGroupComposition` docstring in `sample_representation.py` for details.

This notebook uses the **Human Immune Health Atlas** scRNA-seq object from the Dynamics of Immune Health & Age resource (Nature 2025; PMID [41162704](https://pubmed.ncbi.nlm.nih.gov/41162704/)).

**Where to get the `.h5ad`:** the official download catalog is
[Immune Health Atlas → scRNA-seq Data](https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/downloads/scrna/). The full object is `human_immune_health_atlas_full.h5ad` (~1.8M cells, ~40 GB). Direct GET links that bypass the `explore.allenimmunology.org` IAP redirect often work with `curl` or a browser, e.g. the Allen CDN pattern:

`https://allenimmunology.org/public/publication/download/84792154-cdfb-42d0-8e42-39e210e980b4/filesets/568ad40c-516a-4646-9426-bdcd7029c1f5/human_immune_health_atlas_full.h5ad`

**Local path:** the notebook resolves the file in this order: environment variable `IMMUNE_ATLAS_H5AD`; `IMMUNE_ATLAS_DIR` + `human_immune_health_atlas_full.h5ad`; `docs/notebooks/data/`; then `~/Projects/patient-maps-playground/data/human_immune_health_atlas_full.h5ad` if present; otherwise the small subset `human_immune_health_atlas_other.h5ad` under `docs/notebooks/data/` for smoke tests (see `.gitignore` for `data/`).


## Install patpy

Use a recent `patpy` with `CellGroupComposition` / `GroupedPseudobulk` APIs (development install from the patpy repo is fine).


In [ ]:
!pip install git+https://github.com/lueckenlab/patpy.git@main


In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

import patpy

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
from pathlib import Path

# --- Local AnnData path (full object is ~40 GB; not shipped with this repo) ---
# 1) IMMUNE_ATLAS_H5AD=/abs/path/to/human_immune_health_atlas_full.h5ad
# 2) IMMUNE_ATLAS_DIR=... (directory containing the file)
# 3) ./data/human_immune_health_atlas_full.h5ad next to your notebook cwd
# 4) docs/notebooks/data/... when working inside a patpy clone (submodule layout)
# 5) optional smoke subset human_immune_health_atlas_other.h5ad

NOTEBOOK_DIR = Path.cwd().resolve()
_candidates = []
_env_h5 = os.environ.get("IMMUNE_ATLAS_H5AD")
if _env_h5:
    _candidates.append(Path(_env_h5).expanduser())
_env_dir = os.environ.get("IMMUNE_ATLAS_DIR")
if _env_dir:
    _candidates.append(Path(_env_dir).expanduser() / "human_immune_health_atlas_full.h5ad")

_candidates.append(NOTEBOOK_DIR / "data" / "human_immune_health_atlas_full.h5ad")

if NOTEBOOK_DIR.name == "notebooks" and NOTEBOOK_DIR.parent.name == "tutorials":
    _patpy_docs = NOTEBOOK_DIR.parent.parent
    _candidates.append(
        _patpy_docs / "notebooks" / "data" / "human_immune_health_atlas_full.h5ad"
    )

_candidates.append(
    Path.home()
    / "Projects"
    / "patient-maps-playground"
    / "data"
    / "human_immune_health_atlas_full.h5ad"
)

_OTHER = NOTEBOOK_DIR / "data" / "human_immune_health_atlas_other.h5ad"
if NOTEBOOK_DIR.name == "notebooks" and NOTEBOOK_DIR.parent.name == "tutorials":
    _OTHER = (
        NOTEBOOK_DIR.parent.parent
        / "notebooks"
        / "data"
        / "human_immune_health_atlas_other.h5ad"
    )

ADATA_PATH = None
for pth in _candidates:
    if pth.is_file():
        ADATA_PATH = pth
        break
if ADATA_PATH is None and _OTHER.is_file():
    print(
        "WARNING: full atlas not found; using subset",
        _OTHER.name,
        "for smoke tests only.",
    )
    ADATA_PATH = _OTHER
if ADATA_PATH is None:
    raise FileNotFoundError(
        "No .h5ad found. Download from the Allen Immune Health Atlas (see intro) and set "
        "IMMUNE_ATLAS_H5AD or place human_immune_health_atlas_full.h5ad under ./data (or patpy docs/notebooks/data)."
    )

print("Loading", ADATA_PATH)
adata = sc.read_h5ad(ADATA_PATH)
adata



In [ ]:
# Sample and annotation columns (Allen `.obs` schema)
SAMPLE_KEY = "sample.sampleKitGuid"
RESOLUTIONS = ["AIFI_L1", "AIFI_L2", "AIFI_L3"]
AGE_COL = "sample.subjectAgeAtDraw"
CMV_COL = "subject.cmv"

missing = {c for c in [SAMPLE_KEY, AGE_COL, CMV_COL, *RESOLUTIONS] if c not in adata.obs}
if missing:
    raise ValueError(f"Expected columns not found in obs: {missing}")

if "X_pca" not in adata.obsm:
    raise ValueError(
        "This notebook expects PCA coordinates in adata.obsm['X_pca']. "
        "Run e.g. sc.tl.pca(adata, n_comps=50) if your file differs."
    )

print(adata.obs[SAMPLE_KEY].nunique(), "samples")


In [ ]:
def benchmark_sample_methods(adata, resolutions, n_neighbors: int = 7, random_state: int = 67):
    rows = []
    for res in resolutions:
        method_specs = [
            (
                "composition_clr",
                lambda r=res: patpy.tl.CellGroupComposition(
                    SAMPLE_KEY, r, apply_clr=True, seed=random_state
                ),
            ),
            (
                "composition_prop",
                lambda r=res: patpy.tl.CellGroupComposition(
                    SAMPLE_KEY, r, apply_clr=False, seed=random_state
                ),
            ),
            (
                "pseudobulk",
                lambda r=res: patpy.tl.Pseudobulk(SAMPLE_KEY, r, layer="X_pca", seed=random_state),
            ),
            (
                "grouped_pseudobulk",
                lambda r=res: patpy.tl.GroupedPseudobulk(SAMPLE_KEY, r, layer="X_pca", seed=random_state),
            ),
            (
                "random_baseline",
                lambda r=res: patpy.tl.RandomVector(SAMPLE_KEY, r, latent_dim=32, seed=random_state),
            ),
        ]

        for method_name, factory in method_specs:
            m = factory()
            m.prepare_anndata(adata)
            use_force = method_name.startswith("composition")
            _ = m.calculate_distance_matrix(force=use_force)

            age_res = m.evaluate_representation(
                target=AGE_COL,
                method="knn",
                n_neighbors=n_neighbors,
                task="regression",
            )
            cmv_res = m.evaluate_representation(
                target=CMV_COL,
                method="knn",
                n_neighbors=n_neighbors,
                task="classification",
            )

            rows.append(
                {
                    "resolution": res,
                    "method": method_name,
                    "age_score": age_res["score"],
                    "age_metric": age_res["metric"],
                    "cmv_score": cmv_res["score"],
                    "cmv_metric": cmv_res["metric"],
                }
            )
    return pd.DataFrame(rows)


results_df = benchmark_sample_methods(adata, RESOLUTIONS)
results_df


In [ ]:
# Rank runs: biological covariates are better when score is higher
results_df["mean_bio_score"] = results_df[["age_score", "cmv_score"]].mean(axis=1)
results_sorted = results_df.sort_values("mean_bio_score", ascending=False)
results_sorted.head(8)


In [ ]:
top = results_sorted.iloc[:2]
fitted = {}
for _, r in top.iterrows():
    tag = f"{r['resolution']}::{r['method']}"
    res = r["resolution"]
    name = r["method"]
    if name == "composition_clr":
        m = patpy.tl.CellGroupComposition(SAMPLE_KEY, res, apply_clr=True)
    elif name == "composition_prop":
        m = patpy.tl.CellGroupComposition(SAMPLE_KEY, res, apply_clr=False)
    elif name == "pseudobulk":
        m = patpy.tl.Pseudobulk(SAMPLE_KEY, res, layer="X_pca")
    elif name == "grouped_pseudobulk":
        m = patpy.tl.GroupedPseudobulk(SAMPLE_KEY, res, layer="X_pca")
    else:
        m = patpy.tl.RandomVector(SAMPLE_KEY, res, latent_dim=32)

    m.prepare_anndata(adata)
    m.calculate_distance_matrix(force=name.startswith("composition"))
    fitted[tag] = m
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    m.plot_embedding(method="UMAP", metadata_cols=[AGE_COL, CMV_COL], axes=axes)
    fig.suptitle(tag)
    plt.tight_layout()


## Interpretation

This notebook loads the real **Allen Immune Health Atlas** object (cell-level expression, QC, and AIFI labels) and **benchmarks multiple patpy sample representations**—composition with and without CLR, pseudobulk, grouped pseudobulk, and a random baseline—**separately for each annotation resolution** (`AIFI_L1`–`L3`). KNN scores summarize how well distances recover **age** (regression / Spearman) and **CMV status** (classification / calibrated macro-F1). Choosing the top runs and plotting their UMAP-on-distances embeddings gives a concrete example of how patpy can compare representations across resolutions on a public immune-aging cohort.
